In [4]:
import pandas as pd
import requests
from datetime import datetime, UTC
from urllib.parse import quote
from concurrent.futures import ThreadPoolExecutor, as_completed

INPUT_FILE = "ks.csv"
THREADS = 50   # increase if your network allows

def unix_to_date(ts):
    try:
        if ts in [None, "", 0, "0"]:
            return "NA"
        return datetime.fromtimestamp(int(ts), UTC).strftime("%Y-%m-%d %H:%M:%S")
    except:
        return "NA"

headers = {
    "User-Agent": "Mozilla/5.0"
}

session = requests.Session()
session.headers.update(headers)

def process_row(row):
    host = str(row.host).rstrip("/")
    username = row.username
    password = row.password

    try:
        api_url = f"{host}/player_api.php?username={quote(username)}&password={quote(password)}"

        r = session.get(api_url, timeout=3)
        data = r.json()

        user_info = data.get("user_info", {})
        server_info = data.get("server_info", {})

        status = user_info.get("status")

        if status and status.lower() == "disabled":
            return None

        return {
            "host": host,
            "username": username,
            "password": password,
            "status": status or "NA",
            "created_at": unix_to_date(user_info.get("created_at")),
            "exp_date": unix_to_date(user_info.get("exp_date")),
            "active_cons": user_info.get("active_cons") or "NA",
            "max_connections": user_info.get("max_connections") or "NA",
            "url": server_info.get("url") or "NA",
            "server_protocol": server_info.get("server_protocol") or "NA",
        }

    except:
        return None


# load csv
df = pd.read_csv(INPUT_FILE, dtype=str)

# remove duplicates
df = df.drop_duplicates(subset=["host", "username", "password"])

rows = []

with ThreadPoolExecutor(max_workers=THREADS) as executor:

    futures = [executor.submit(process_row, r) for r in df.itertuples(index=False)]

    for f in as_completed(futures):
        result = f.result()
        if result:
            rows.append(result)


out_df = pd.DataFrame(rows)

# convert for sorting
out_df["active_cons"] = pd.to_numeric(out_df["active_cons"], errors="coerce")
out_df["max_connections"] = pd.to_numeric(out_df["max_connections"], errors="coerce")
out_df["created_at"] = pd.to_datetime(out_df["created_at"], errors="coerce")
out_df["exp_date"] = pd.to_datetime(out_df["exp_date"], errors="coerce")

# sort
out_df = out_df.sort_values(
    by=["host", "active_cons", "max_connections", "created_at", "exp_date"],
    ascending=[True, True, False, True, False]
)

# convert back
out_df["created_at"] = out_df["created_at"].astype(str)
out_df["exp_date"] = out_df["exp_date"].astype(str)

# overwrite
out_df.to_csv(INPUT_FILE, index=False)

/tmp/ipykernel_6846/1888350068.py:85: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  out_df["exp_date"] = pd.to_datetime(out_df["exp_date"], errors="coerce")
